# LeetCode 422 — Valid Word Square bằng Python

**Link đề:** https://leetcode.com/problems/valid-word-square/
**Độ khó:** Easy · **Chủ đề:** Array, Matrix

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def validWordSquare(self, words: List[str]) -> bool:
        ...
```

## Mục tiêu học
1. Hiểu **ma trận** (matrix) dạng list chuỗi: hàng `k` là `words[k]`, cột `k` là các ký tự `words[r][k]`.
2. Hiểu phép **chuyển vị** (transpose) và tính **đối xứng qua đường chéo chính**: `words[i][j] == words[j][i]`.
3. Xử lý **hàng có độ dài khác nhau** (ma trận "răng cưa" — jagged) mà không bị `IndexError`.
4. Viết 4 lời giải: dựng từng cột rồi so (baseline) → so từng ô với ô đối xứng (chính) → `zip_longest(*words)` chuyển vị → đệm thành ma trận vuông.
5. Trace, chứng minh, Time/Space, lỗi hay gặp, biến thể.

## 1. Đề bài

Cho mảng chuỗi `words`. Trả về `True` nếu nó tạo thành một **hình vuông từ hợp lệ** (valid word square).

Một dãy chuỗi là word square hợp lệ nếu **hàng thứ `k` và cột thứ `k` đọc ra cùng một chuỗi**, với mọi `0 <= k < max(số hàng, số cột)`.

**Ví dụ 1:** `["abcd","bnrt","crmy","dtye"]` → `True`

```text
a b c d
b n r t
c r m y
d t y e
```
Hàng 0 = `abcd` = cột 0; hàng 1 = `bnrt` = cột 1; ...

**Ví dụ 2:** `["abcd","bnrt","crm","dt"]` → `True` (các hàng dài khác nhau nhưng vẫn khớp)

```text
a b c d
b n r t
c r m
d t
```

**Ví dụ 3:** `["ball","area","read","lady"]` → `False` — hàng 2 là `"read"` nhưng cột 2 là `"lead"`.

**Constraints:**
- `1 <= words.length <= 500`
- `1 <= words[i].length <= 500`
- `words[i]` chỉ gồm chữ cái tiếng Anh thường.

## 2. Nền tảng Python

### 2.1. List chuỗi như một ma trận

`words[r]` là hàng `r` (một chuỗi), `words[r][c]` là ký tự ở hàng `r`, cột `c`. Mỗi hàng có thể dài khác nhau.

In [1]:
from typing import List

words = ["abcd", "bnrt", "crm", "dt"]
print("số hàng:", len(words))
print("độ dài từng hàng:", [len(w) for w in words])
print("words[2] =", words[2], "| words[2][1] =", words[2][1])
for r, w in enumerate(words):
    print(r, " ".join(w))

số hàng: 4
độ dài từng hàng: [4, 4, 3, 2]
words[2] = crm | words[2][1] = r
0 a b c d
1 b n r t
2 c r m
3 d t


`enumerate(words)` trả lần lượt các cặp `(chỉ_số, phần_tử)` — tiện khi cần cả hai. `" ".join(w)` chèn dấu cách giữa các ký tự để in như lưới.

### 2.2. Đọc **cột** `k`: lấy ký tự `k` của mọi hàng **đủ dài**

Hàng `r` chỉ có ký tự ở cột `k` nếu `k < len(words[r])`. Hàng ngắn hơn thì "trống" ở cột đó.

In [2]:
def column(words: List[str], k: int) -> str:
    return "".join(w[k] for w in words if k < len(w))

for k in range(4):
    print(f"cột {k}: {column(words, k)!r} | hàng {k}: {words[k]!r}")

cột 0: 'abcd' | hàng 0: 'abcd'
cột 1: 'bnrt' | hàng 1: 'bnrt'
cột 2: 'crm' | hàng 2: 'crm'
cột 3: 'dt' | hàng 3: 'dt'


Chú ý: cột của ma trận răng cưa có thể có "lỗ" ở giữa (một hàng ngắn kẹp giữa hai hàng dài). Hàm `column` ở trên **bỏ qua lỗ**, nên có thể nối hai ký tự không liền nhau — ta sẽ quay lại điểm này ở phần lỗi hay gặp.

### 2.3. Chuyển vị (transpose) và đối xứng qua đường chéo

Chuyển vị ma trận: hàng thành cột. Ô `(i, j)` chuyển sang `(j, i)`. "Hàng `k` = cột `k` với mọi `k`" chính là **ma trận bằng chuyển vị của nó** ⟺ `M[i][j] == M[j][i]` với mọi `i, j` — đối xứng qua **đường chéo chính** (các ô `(k, k)`).

In [3]:
M = ["abc",
     "bde",
     "cef"]
for i in range(3):
    print([(M[i][j], M[j][i]) for j in range(3)])
print("đối xứng:", all(M[i][j] == M[j][i] for i in range(3) for j in range(3)))

[('a', 'a'), ('b', 'b'), ('c', 'c')]
[('b', 'b'), ('d', 'd'), ('e', 'e')]
[('c', 'c'), ('e', 'e'), ('f', 'f')]
đối xứng: True


`all(...)` trả `True` khi **mọi** phần tử là đúng (và dừng sớm khi gặp phần tử sai). Biểu thức `for i ... for j ...` lồng hai vòng trong một generator.

## 3. Cách trực giác (baseline) — dựng từng cột rồi so với hàng

Làm đúng theo định nghĩa: với mỗi `k` từ 0 đến `max(số hàng, số cột) - 1`, dựng **hàng k** (hoặc `""` nếu không có) và **cột k**, so sánh.

Để tránh vấn đề "lỗ", ta dựng cột cẩn thận: duyệt hàng từ trên xuống, **dừng** ở hàng đầu tiên quá ngắn; nếu sau đó vẫn còn hàng đủ dài thì cột có lỗ → không thể bằng một hàng (hàng không có lỗ) → `False`.

In [4]:
def column_strict(words: List[str], k: int):
    # Trả về cột k nếu cột liền mạch, None nếu cột có lỗ.
    chars = []
    ended = False
    for w in words:
        if k < len(w):
            if ended:
                return None          # có lỗ: hàng ngắn kẹp giữa
            chars.append(w[k])
        else:
            ended = True
    return "".join(chars)


def valid_word_square_columns(words: List[str]) -> bool:
    n_rows = len(words)
    n_cols = max(len(w) for w in words)
    for k in range(max(n_rows, n_cols)):
        row = words[k] if k < n_rows else ""
        col = column_strict(words, k)
        if col is None or row != col:
            return False
    return True

print(valid_word_square_columns(["abcd", "bnrt", "crmy", "dtye"]))   # True
print(valid_word_square_columns(["abcd", "bnrt", "crm", "dt"]))      # True
print(valid_word_square_columns(["ball", "area", "read", "lady"]))   # False

True
True
False


## 4. Nút thắt và insight

**Nút thắt:** baseline dựng **mỗi** cột bằng cách đi qua **mọi** hàng → `O(K · n)` với `K = max(số hàng, số cột)`, và tạo nhiều chuỗi tạm. Ta lặp lại công việc: ô `(i, j)` được đọc khi dựng cột `j`, rồi lại so khi xét hàng `i`.

**Insight:** "hàng `k` = cột `k` với mọi `k`" ⟺ **mỗi ô có ô đối xứng và bằng nó**:

> với mọi `i`, mọi `j < len(words[i])`: phải có `j < len(words)`, `i < len(words[j])`, và `words[i][j] == words[j][i]`.

Chỉ cần duyệt **mỗi ô đúng một lần**, không tạo chuỗi phụ. Điều kiện về chỉ số chính là bắt "độ dài không khớp": nếu ô `(i, j)` tồn tại mà ô `(j, i)` không tồn tại → hàng `j` ngắn hơn cột `j` → `False`.

In [5]:
words = ["ball", "area", "read", "lady"]
i, j = 0, 1
print(f"words[{i}][{j}] = {words[i][j]!r} vs words[{j}][{i}] = {words[j][i]!r}  <- bằng nhau")
i, j = 2, 0
print(f"words[{i}][{j}] = {words[i][j]!r} vs words[{j}][{i}] = {words[j][i]!r}  <- khác nhau!")
print("hàng 2:", words[2], "| cột 2:", "".join(w[2] for w in words))

words[0][1] = 'a' vs words[1][0] = 'a'  <- bằng nhau
words[2][0] = 'r' vs words[0][2] = 'l'  <- khác nhau!
hàng 2: read | cột 2: lead


## 5. Lời giải chính — so từng ô với ô đối xứng

**Đoạn 1 — duyệt mọi ô tồn tại:**
```python
for i in range(len(words)):
    for j in range(len(words[i])):
```
**Đoạn 2 — ô đối xứng phải tồn tại:**
```python
        if j >= len(words) or i >= len(words[j]):
            return False
```
**Đoạn 3 — và phải bằng nhau:**
```python
        if words[i][j] != words[j][i]:
            return False
```
**Đoạn 4 —** không vi phạm nào → `return True`.

Vì sao không cần kiểm tra ngược (ô `(j, i)` tồn tại mà ô `(i, j)` không)? Vì vòng lặp cũng sẽ đi qua ô `(j, i)` và khi đó kiểm tra chính ô `(i, j)`. Mọi ô được "đối chất" với ô đối xứng.

In [6]:
def valid_word_square_cells(words: List[str]) -> bool:
    for i in range(len(words)):
        for j in range(len(words[i])):
            if j >= len(words) or i >= len(words[j]):
                return False
            if words[i][j] != words[j][i]:
                return False
    return True

print(valid_word_square_cells(["abcd", "bnrt", "crmy", "dtye"]))   # True
print(valid_word_square_cells(["abcd", "bnrt", "crm", "dt"]))      # True
print(valid_word_square_cells(["ball", "area", "read", "lady"]))   # False
print(valid_word_square_cells(["abc", "b"]))                       # False: cột 0 = "ab" nhưng hàng 0 = "abc"

True
True
False
False


## 6. TRACE từng bước

Trace ví dụ 3 `["ball","area","read","lady"]`:

| i | j | words[i][j] | words[j][i] | Kết luận |
|:---:|:---:|:---:|:---:|---|
| 0 | 0 | b | b | đường chéo, khớp |
| 0 | 1 | a | a (`words[1][0]`) | khớp |
| 0 | 2 | l | r (`words[2][0]`) | **khác** → `False` |

Thuật toán dừng ngay ở ô `(0, 2)`. Cặp ô `(0,2)`/`(2,0)` chính là ký tự **đầu** của hàng 2 (`"read"` → `r`) và của cột 2 (`"lead"` → `l`) — đúng lý do đề nêu. (Để ý: cặp ô này cũng làm hàng 0 `"ball"` khác cột 0 `"barl"`.)

In [7]:
def trace_word_square(words: List[str]) -> bool:
    for i in range(len(words)):
        for j in range(len(words[i])):
            if j >= len(words) or i >= len(words[j]):
                print(f"  ô ({i},{j})='{words[i][j]}' nhưng ô đối xứng ({j},{i}) KHÔNG tồn tại -> False")
                return False
            a, b_ = words[i][j], words[j][i]
            mark = "==" if a == b_ else "!="
            print(f"  ({i},{j})='{a}' {mark} ({j},{i})='{b_}'")
            if a != b_:
                print("  -> False")
                return False
    print("  không có vi phạm -> True")
    return True

print("Ví dụ 3:")
trace_word_square(["ball", "area", "read", "lady"])
print("\nThiếu ô đối xứng:")
trace_word_square(["abc", "b"])

Ví dụ 3:
  (0,0)='b' == (0,0)='b'
  (0,1)='a' == (1,0)='a'
  (0,2)='l' != (2,0)='r'
  -> False

Thiếu ô đối xứng:
  (0,0)='a' == (0,0)='a'
  (0,1)='b' == (1,0)='b'
  ô (0,2)='c' nhưng ô đối xứng (2,0) KHÔNG tồn tại -> False


False

### 6.1. Trace ví dụ 2 (hàng răng cưa nhưng hợp lệ)

Mỗi ô `(i, j)` đều có ô `(j, i)`: ví dụ `(0,3)='d'` ↔ `(3,0)='d'`; `(2,2)='m'` nằm trên đường chéo, tự so với chính nó.

In [8]:
trace_word_square(["abcd", "bnrt", "crm", "dt"])

  (0,0)='a' == (0,0)='a'
  (0,1)='b' == (1,0)='b'
  (0,2)='c' == (2,0)='c'
  (0,3)='d' == (3,0)='d'
  (1,0)='b' == (0,1)='b'
  (1,1)='n' == (1,1)='n'
  (1,2)='r' == (2,1)='r'
  (1,3)='t' == (3,1)='t'
  (2,0)='c' == (0,2)='c'
  (2,1)='r' == (1,2)='r'
  (2,2)='m' == (2,2)='m'
  (3,0)='d' == (0,3)='d'
  (3,1)='t' == (1,3)='t'
  không có vi phạm -> True


True

## 7. Các lời giải khác

### 7.1. Cách 3 — chuyển vị bằng `zip_longest(*words)`

`zip(*words)` "xoay" ma trận: phần tử thứ `k` của kết quả là bộ ký tự cột `k`. Nhưng `zip` **cắt** theo hàng ngắn nhất. `itertools.zip_longest(*words, fillvalue="")` thì điền `""` cho hàng thiếu → cột `k` = `"".join(...)`.

Dấu `*words` "rải" list thành nhiều đối số: `zip_longest(*["ab","cd"])` = `zip_longest("ab", "cd")`.

Ma trận hợp lệ ⟺ danh sách cột (sau khi ghép) **bằng** danh sách hàng.

In [9]:
from itertools import zip_longest

print(list(zip(*["abc", "de"])))
print(list(zip_longest(*["abc", "de"], fillvalue="")))

[('a', 'd'), ('b', 'e')]
[('a', 'd'), ('b', 'e'), ('c', '')]


In [10]:
def valid_word_square_zip(words: List[str]) -> bool:
    columns = ["".join(col) for col in zip_longest(*words, fillvalue="")]
    return columns == words

print(valid_word_square_zip(["abcd", "bnrt", "crmy", "dtye"]))
print(valid_word_square_zip(["abcd", "bnrt", "crm", "dt"]))
print(valid_word_square_zip(["ball", "area", "read", "lady"]))
print(valid_word_square_zip(["abc", "b"]))

True
True
False
False


**Cẩn thận "lỗ":** `zip_longest` điền `""` vào lỗ nên có thể **ghép** hai ký tự không liền nhau thành một cột trông "liền mạch". Vì sao cách này vẫn đúng? Nếu `columns == words` thì độ dài cột `k` bằng độ dài hàng `k`. Độ dài cột `k` = **số hàng dài hơn `k`**, là dãy **không tăng** theo `k` ⇒ độ dài các hàng cũng không tăng từ trên xuống ⇒ không thể có lỗ. Mục 11 còn **kiểm thử đối chiếu** cách này với cách so từng ô trên hàng nghìn ma trận ngẫu nhiên.

### 7.2. Cách 4 — đệm thành ma trận vuông

Đặt `K = max(số hàng, độ dài hàng dài nhất)`. Đệm mỗi hàng bằng ký tự `' '` cho đủ `K`, thêm hàng toàn `' '` nếu thiếu hàng. Khi đó lưới là `K × K` đầy đủ và chỉ cần kiểm tra `grid[i][j] == grid[j][i]`. Ký tự đệm (dấu cách) không xuất hiện trong input, nên "ô trống" chỉ bằng "ô trống".

`str.ljust(K)` đệm dấu cách bên phải cho đủ `K` ký tự.

In [11]:
print(repr("ab".ljust(4)))

def valid_word_square_pad(words: List[str]) -> bool:
    K = max(len(words), max(len(w) for w in words))
    grid = [w.ljust(K) for w in words] + [" " * K] * (K - len(words))
    for i in range(K):
        for j in range(i + 1, K):          # chỉ cần nửa trên đường chéo
            if grid[i][j] != grid[j][i]:
                return False
    return True

print(valid_word_square_pad(["abcd", "bnrt", "crm", "dt"]), valid_word_square_pad(["ball", "area", "read", "lady"]))

'ab  '
True False


Mẹo `range(i + 1, K)`: vì so `(i, j)` với `(j, i)` là đối xứng, chỉ cần duyệt **nửa trên** đường chéo (`j > i`) — số phép so giảm một nửa. Đường chéo `(k, k)` tự bằng chính nó.

## 8. Code nộp LeetCode

Bản chính: so từng ô (không tạo chuỗi phụ, `O(1)` bộ nhớ). Bản phụ: `zip_longest` một dòng.

In [12]:
class Solution:
    def validWordSquare(self, words: List[str]) -> bool:
        n = len(words)
        for i in range(n):
            for j in range(len(words[i])):
                if j >= n or i >= len(words[j]) or words[i][j] != words[j][i]:
                    return False
        return True


sol = Solution()
print(sol.validWordSquare(["abcd", "bnrt", "crmy", "dtye"]))
print(sol.validWordSquare(["abcd", "bnrt", "crm", "dt"]))
print(sol.validWordSquare(["ball", "area", "read", "lady"]))

True
True
False


In [13]:
class SolutionZip:
    def validWordSquare(self, words: List[str]) -> bool:
        return ["".join(c) for c in zip_longest(*words, fillvalue="")] == words

print(SolutionZip().validWordSquare(["abcd", "bnrt", "crm", "dt"]))

True


Điều kiện gộp `j >= n or i >= len(words[j]) or ...` dựa vào **đánh giá ngắn mạch** (short-circuit): `or` dừng ngay khi gặp vế đúng, nên `words[j]` chỉ được đọc khi `j < n`, và `words[j][i]` chỉ khi `i < len(words[j])`. Thứ tự các vế **quan trọng**.

## 9. Vì sao đúng?

Coi `words` là tập ô `C = {(i, j) : 0 <= i < n, 0 <= j < len(words[i])}` với nhãn `words[i][j]`.

**Định nghĩa** "hàng k = cột k với mọi k" ⟺ (1) với mọi `k`, độ dài hàng `k` bằng độ dài cột `k`, các ký tự lần lượt bằng nhau. Điều này tương đương với:

> (★) `(i, j) ∈ C` ⟺ `(j, i) ∈ C`, và khi đó `words[i][j] == words[j][i]`.

- (★) ⇒ hợp lệ: hàng `k` gồm các ô `(k, 0..L-1)`. Theo (★) các ô `(0..L-1, k)` đều tồn tại và cùng nhãn; và không có ô `(L, k)` (vì `(k, L)` không tồn tại). Do mọi hàng đều liền mạch từ cột 0, cột `k` cũng liền mạch → cột `k` = hàng `k`.
- Hợp lệ ⇒ (★): nếu ô `(i, j)` tồn tại thì ký tự thứ `i` của cột `j` tồn tại; cột `j` = hàng `j` nên `words[j]` có ký tự thứ `i` và bằng nhau.

Thuật toán kiểm tra đúng (★): duyệt mọi `(i, j) ∈ C`, đòi `(j, i) ∈ C` và bằng nhãn. Chiều ngược của "⟺" được kiểm tra khi vòng lặp tới chính ô `(j, i)`. ∎

## 10. Độ phức tạp

Gọi `n` = số hàng, `L` = độ dài hàng dài nhất, `S` = tổng số ký tự (`S <= n · L`), `K = max(n, L)`.

| Cách | Time | Space phụ |
|---|:---:|:---:|
| Dựng từng cột | `O(K · n)` | `O(n)` cho mỗi cột tạm |
| **So từng ô** | `O(S)` | `O(1)` |
| `zip_longest` | `O(n · L)` | `O(n · L)` cho bảng chuyển vị |
| Đệm vuông | `O(K²)` | `O(K²)` |

Với giới hạn `500 × 500`, mọi cách đều chạy nhanh, nhưng cách so từng ô là gọn nhất về bộ nhớ và dừng sớm khi gặp vi phạm.

## 11. Bộ test

In [14]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS  {name:<36} -> {actual}")

sol = Solution()
cases = [
    ("đề 1", ["abcd", "bnrt", "crmy", "dtye"], True),
    ("đề 2: răng cưa hợp lệ", ["abcd", "bnrt", "crm", "dt"], True),
    ("đề 3: read vs lead", ["ball", "area", "read", "lady"], False),
    ("một ký tự", ["a"], True),
    ("một hàng dài", ["abc"], False),
    ("một cột dài", ["a", "b", "c"], False),
    ("hàng dài hơn số hàng", ["abc", "b"], False),
    ("hàng ngắn vẫn hợp lệ", ["ab", "b"], True),
    ("tam giác hợp lệ", ["abc", "b", "c"], True),
    ("lỗ trong cột", ["abc", "b", "cd"], False),
    ("đối xứng nhưng thừa hàng", ["ab", "b", "c"], False),
]
for name, words, exp in cases:
    check(name, sol.validWordSquare(words), exp)

PASS  đề 1                                 -> True
PASS  đề 2: răng cưa hợp lệ                -> True
PASS  đề 3: read vs lead                   -> False
PASS  một ký tự                            -> True
PASS  một hàng dài                         -> False
PASS  một cột dài                          -> False
PASS  hàng dài hơn số hàng                 -> False
PASS  hàng ngắn vẫn hợp lệ                 -> True
PASS  tam giác hợp lệ                      -> True
PASS  lỗ trong cột                         -> False
PASS  đối xứng nhưng thừa hàng             -> False


In [15]:
import random
rng = random.Random(422)

def random_words(rng):
    # Sinh ma trận: một nửa là đối xứng thật (rồi có thể làm hỏng), một nửa ngẫu nhiên hoàn toàn.
    n = rng.randint(1, 5)
    if rng.random() < 0.5:
        lens = sorted([rng.randint(1, n) for _ in range(n)], reverse=True)
        lens[0] = n
        grid = [[None] * lens[i] for i in range(n)]
        ok = all(lens[j] > i for i in range(n) for j in range(lens[i]))   # hình dạng tự đối xứng?
        for i in range(n):
            for j in range(lens[i]):
                if grid[i][j] is None:
                    ch = rng.choice("ab")
                    grid[i][j] = ch
                    if ok and j < n and i < lens[j]:
                        grid[j][i] = ch
        words = ["".join(r) for r in grid]
        if rng.random() < 0.3:
            r = rng.randrange(n)
            words[r] = words[r][:-1] + rng.choice("ab") if len(words[r]) > 1 else words[r]
        return words
    return ["".join(rng.choice("ab") for _ in range(rng.randint(1, 5))) for _ in range(n)]

funcs = [valid_word_square_columns, valid_word_square_cells, valid_word_square_zip,
         valid_word_square_pad, Solution().validWordSquare, SolutionZip().validWordSquare]
trues = 0
for _ in range(5000):
    w = random_words(rng)
    exp = valid_word_square_cells(w)
    trues += exp
    for f in funcs:
        assert f(w) == exp, (f.__name__, w)
print(f"5000 ma trận ngẫu nhiên ({trues} hợp lệ): 6 cách đồng thuận -> PASS")

5000 ma trận ngẫu nhiên (1824 hợp lệ): 6 cách đồng thuận -> PASS


Kết quả đối chiếu cho thấy cách `zip_longest` cũng đúng: khi cột có "lỗ", cột ghép lại có **ít ký tự hơn** hàng tương ứng hoặc lệch vị trí, nên `columns == words` vẫn bắt được. Tuy vậy với người mới, cách so từng ô dễ **chứng minh** hơn.

## 12. Lỗi người mới hay gặp

### Lỗi 1: Giả sử ma trận vuông đầy đủ
```python
for i in range(n):
    for j in range(n):
        if words[i][j] != words[j][i]: ...
```
Hàng ngắn → `IndexError` (ví dụ 2 của đề có hàng `"dt"`).

In [16]:
def wrong_assume_square(words):
    n = len(words)
    for i in range(n):
        for j in range(n):
            if words[i][j] != words[j][i]:
                return False
    return True

try:
    wrong_assume_square(["abcd", "bnrt", "crm", "dt"])
except IndexError as e:
    print("IndexError:", e)

IndexError: string index out of range


### Lỗi 2: Chỉ kiểm tra ký tự, quên kiểm tra **tồn tại** của ô đối xứng (dùng `try/except` nuốt lỗi)

```python
try:
    if words[i][j] != words[j][i]: return False
except IndexError:
    continue              # bỏ qua -> sai!
```
`["abc", "b"]`: ô `(0, 2)` không có đối xứng `(2, 0)` → bị bỏ qua → trả `True` sai.

### Lỗi 3: Dùng `zip` thay vì `zip_longest`
`zip` cắt theo hàng ngắn nhất → mất ký tự → `["abc","b","c"]`... bị đánh giá sai.

In [17]:
def wrong_swallow(words):
    for i in range(len(words)):
        for j in range(len(words[i])):
            try:
                if words[i][j] != words[j][i]:
                    return False
            except IndexError:
                continue
    return True

def wrong_zip(words):
    return ["".join(c) for c in zip(*words)] == words

print("lỗi 2 ['abc','b']:", wrong_swallow(["abc", "b"]), "(đúng False)")
print("lỗi 3 ['abc','b','c']:", wrong_zip(["abc", "b", "c"]), "(đúng True)")

lỗi 2 ['abc','b']: True (đúng False)
lỗi 3 ['abc','b','c']: False (đúng True)


### Lỗi 4: Sai thứ tự điều kiện trong `or`
```python
if words[i][j] != words[j][i] or j >= n: ...
```
Đọc `words[j]` **trước** khi kiểm tra `j < n` → `IndexError`. Luôn đặt kiểm tra chỉ số **trước**.

### Lỗi 5: Chỉ so `len(words) == len(words[0])`
Đúng số hàng = số cột của hàng đầu là **cần** nhưng chưa **đủ** (ví dụ 3 của đề vẫn 4×4 mà sai).

## 13. Mẹo nhận dạng pattern

- "Hàng k = cột k", "đối xứng", "ma trận bằng chuyển vị" → so `M[i][j]` với `M[j][i]`; chỉ cần nửa trên đường chéo khi ma trận vuông đầy đủ.
- Hàng dài khác nhau (jagged) → luôn kiểm tra chỉ số trước khi truy cập; hoặc đệm thành ma trận đầy đủ.
- Chuyển vị trong Python → `zip(*M)` (cắt ngắn) / `zip_longest(*M, fillvalue=...)` (giữ đủ).

Bài liên quan: 867 Transpose Matrix, 766 Toeplitz Matrix (so `M[i][j]` với `M[i-1][j-1]`), 425 Word Squares (Hard — sinh mọi word square).

## 14. Biến thể

1. **LeetCode 867 — Transpose Matrix** (ma trận số đầy đủ).
2. **Ma trận đối xứng số** (`n × n`): kiểm tra `A[i][j] == A[j][i]`.
3. **LeetCode 766 — Toeplitz Matrix:** mỗi đường chéo từ trên-trái xuống dưới-phải là hằng số.

In [18]:
def transpose(matrix):
    return [list(row) for row in zip(*matrix)]

def is_toeplitz(matrix):
    for i in range(1, len(matrix)):
        for j in range(1, len(matrix[0])):
            if matrix[i][j] != matrix[i - 1][j - 1]:
                return False
    return True

print(transpose([[1, 2, 3], [4, 5, 6]]))
print(is_toeplitz([[1, 2, 3, 4], [5, 1, 2, 3], [9, 5, 1, 2]]), is_toeplitz([[1, 2], [2, 2]]))

[[1, 4], [2, 5], [3, 6]]
True False


## 15. Bài tập (TODO)

**Bài 1 (tính tay):** `["abc","bde","cef"]` có hợp lệ không? Viết ra 3 hàng và 3 cột.

**Bài 2 (đoán output):** `Solution().validWordSquare(["ab","bc","c"])`? Gợi ý: ô `(1,1)='c'` thì sao, và hàng 2 dài bao nhiêu so với cột 2?

**Bài 3 (điền chỗ trống):** hoàn thiện cách so từng ô.

In [19]:
def exercise_valid_word_square(words: List[str]) -> bool:
    n = len(words)
    for i in range(n):
        for j in range(len(words[i])):
            # TODO 1: ô đối xứng (j, i) có tồn tại không?
            # TODO 2: hai ký tự có bằng nhau không?
            pass
    # TODO 3: không vi phạm -> True
    pass

# assert exercise_valid_word_square(["abcd", "bnrt", "crm", "dt"]) is True
# assert exercise_valid_word_square(["ball", "area", "read", "lady"]) is False

**Bài 4 (tự viết):** `exercise_first_bad_row(words)` trả về **chỉ số `k` nhỏ nhất** mà hàng `k` khác cột `k` (dùng định nghĩa hàng/cột; hàng không tồn tại coi là `""`), hoặc `-1` nếu hợp lệ.

**Bài 5 (biến thể):** `exercise_is_symmetric(matrix)` cho ma trận số **vuông đầy đủ**, chỉ duyệt nửa trên đường chéo.

**Bài 6 (mini challenge):** `exercise_make_square(word_list)` — cho đúng `n` từ độ dài `n`, hãy kiểm tra xem có **hoán vị thứ tự hàng** nào tạo thành word square hợp lệ không; trả về một thứ tự hợp lệ (list từ) hoặc `None`. (`n <= 6`, có thể dùng `itertools.permutations`.)

In [20]:
def exercise_first_bad_row(words: List[str]) -> int:
    # TODO
    pass

def exercise_is_symmetric(matrix: List[List[int]]) -> bool:
    # TODO
    pass

def exercise_make_square(word_list: List[str]):
    # TODO
    pass

# assert exercise_first_bad_row(["abcd", "bnrt", "crmz", "dtye"]) == 2
# assert exercise_is_symmetric([[1, 2], [2, 3]]) is True
# assert exercise_make_square(["area", "lady", "ball", "lead"]) == ["ball", "area", "lead", "lady"]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Đọc cột bằng ký tự thứ k mỗi hàng | Cột 0 = `abc` | So `M[i][j]` và `M[j][i]` |
| 2 | Hàng 2 = `"c"` | Cột 2 = ký tự 2 của mọi hàng đủ dài | Hàng 0 có độ dài 2 → cột 2 chỉ có ... |
| 3 | Theo mục 5 | Kiểm `j >= n` trước | `or i >= len(words[j])` rồi mới so ký tự |
| 4 | Dùng hàm `column_strict` | Duyệt `k` tới `max(n, L)` | Cột có lỗ (`None`) → trả `k` |
| 5 | Ma trận vuông | `for j in range(i + 1, n)` | `A[i][j] != A[j][i]` → False |
| 6 | Thử mọi thứ tự | `permutations(word_list)` | `if Solution().validWordSquare(list(p)): return list(p)` |

## 17. Đáp án tham khảo

- **Bài 1:** hàng `abc, bde, cef`; cột 0 = `abc`, cột 1 = `bde`, cột 2 = `cef` → **True**.
- **Bài 2:** ô `(1,1)='c'` tự khớp; hàng 2 = `"c"`, cột 2 = ký tự thứ 2 của các hàng đủ dài — không hàng nào dài 3 → cột 2 rỗng ≠ `"c"`. Trong cách so ô: ô `(2,0)='c'` đòi ô `(0,2)` nhưng `"ab"` chỉ dài 2 → **False**.

In [21]:
print("Bài 1:", Solution().validWordSquare(["abc", "bde", "cef"]))
print("Bài 2:")
trace_word_square(["ab", "bc", "c"])

Bài 1: True
Bài 2:
  (0,0)='a' == (0,0)='a'
  (0,1)='b' == (1,0)='b'
  (1,0)='b' == (0,1)='b'
  (1,1)='c' == (1,1)='c'
  ô (2,0)='c' nhưng ô đối xứng (0,2) KHÔNG tồn tại -> False


False

In [22]:
from itertools import permutations

def answer_valid_word_square(words: List[str]) -> bool:
    n = len(words)
    for i in range(n):
        for j in range(len(words[i])):
            if j >= n or i >= len(words[j]):
                return False
            if words[i][j] != words[j][i]:
                return False
    return True

def answer_first_bad_row(words: List[str]) -> int:
    n = len(words)
    L = max(len(w) for w in words)
    for k in range(max(n, L)):
        row = words[k] if k < n else ""
        col = column_strict(words, k)
        if col is None or row != col:
            return k
    return -1

def answer_is_symmetric(matrix: List[List[int]]) -> bool:
    n = len(matrix)
    for i in range(n):
        for j in range(i + 1, n):
            if matrix[i][j] != matrix[j][i]:
                return False
    return True

def answer_make_square(word_list: List[str]):
    for p in permutations(word_list):
        if answer_valid_word_square(list(p)):
            return list(p)
    return None

assert answer_valid_word_square(["abcd", "bnrt", "crm", "dt"]) is True
assert answer_valid_word_square(["ball", "area", "read", "lady"]) is False
assert answer_first_bad_row(["ball", "area", "read", "lady"]) == 0     # cột 0 là "barl"
assert answer_first_bad_row(["abcd", "bnrt", "crmz", "dtye"]) == 2
assert answer_first_bad_row(["abcd", "bnrt", "crm", "dt"]) == -1
assert answer_first_bad_row(["abc", "b"]) == 0
assert answer_is_symmetric([[1, 2], [2, 3]]) is True and answer_is_symmetric([[1, 2], [3, 1]]) is False
assert answer_make_square(["area", "lady", "ball", "lead"]) == ["ball", "area", "lead", "lady"]
assert answer_make_square(["ab", "cd"]) is None
print("Đáp án bài 3–6 PASS")

Đáp án bài 3–6 PASS


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def validWordSquare(self, words: List[str]) -> bool:
        n = len(words)
        for i in range(n):
            for j in range(len(words[i])):
                # ô đối xứng phải tồn tại VÀ bằng nhau (kiểm chỉ số trước!)
                if j >= n or i >= len(words[j]) or words[i][j] != words[j][i]:
                    return False
        return True
```

**Checklist:**
- [ ] Không giả sử ma trận vuông / các hàng dài bằng nhau.
- [ ] Kiểm tra `j < len(words)` và `i < len(words[j])` **trước** khi đọc `words[j][i]`.
- [ ] Không dùng `try/except` để nuốt lỗi chỉ số.
- [ ] Nếu chuyển vị: dùng `zip_longest`, không dùng `zip`.
- [ ] Time `O(S)` (tổng số ký tự), Space `O(1)`.